# 🤗 Hugging Face Repository Browser

In [3]:
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
from huggingface_hub import list_repo_files

# UIを作成
repo_input = widgets.Text(
    value='nvidia/NVIDIA-Nemotron-Parse-v1.1',
    placeholder='Example: nvidia/NVIDIA-Nemotron-Parse-v1.1',
    description='Repository Name:',
    style={'description_width': 'initial'}
)

button = widgets.Button(description='🔍 Fetching file list...')
output = widgets.Output()
download_output = widgets.Output()

# チェックボックスとダウンロードコマンドを保存する変数
file_checkboxes = []
download_commands = []

# ダウンロードコマンド生成関数
def generate_download_commands(repo_id, selected_files):
    base_url = f"https://huggingface.co/{repo_id}/resolve/main/"
    
    wget_cmds = [f"mkdir -p '{repo_id}'", f"cd '{repo_id}'"]
    curl_cmds = [f"mkdir -p '{repo_id}'", f"cd '{repo_id}'"]
    
    for file in selected_files:
        if file.strip():  # 空でない場合
            url = base_url + file
            
            parts = file.split('/')
            depth = len(parts) - 1
            
            wget_cmd = f"wget '{url}'" + (f" -P '{"/".join(parts[:-1])}'" if depth > 0 else "")
            curl_cmd = f"curl -L -o '{file}' '{url}' --create-dirs"
            
            wget_cmds.append(wget_cmd)
            curl_cmds.append(curl_cmd)
    
    return '\n'.join(wget_cmds), '\n'.join(curl_cmds)

# ツリー表示とチェックボックス作成関数
def create_tree_with_checkboxes(files):
    if not files:
        return "<p>No files.</p>", []
    
    # パスでグループ化
    path_groups = {}
    checkbox_widgets = []
    
    for file in files:
        parts = file.split('/')
        current_path = ''
        
        # フォルダ構造を作成
        for i, part in enumerate(parts[:-1]):
            current_path += part + '/'
            if current_path not in path_groups:
                path_groups[current_path] = []
        
        if file not in path_groups:
            path_groups[file] = []
    
    # チェックボックスとHTMLを生成
    html = "<div style='font-family: monospace;'>"
    
    # フォルダを先に表示
    for path in sorted(path_groups.keys()):
        parts = path.split('/')
        depth = len(parts) - 1
        indent = '　' * depth
        
        if path.endswith('/'):
            folder_name = parts[-1] or 'root'
            html += f"<div style='color: #00d4aa; font-weight: bold;'>{indent}📁 {folder_name}/</div>"
    
    # ファイル用のチェックボックスを作成
    for file in sorted(files):
        if not file.endswith('/'):  # フォルダでない場合のみ
            parts = file.split('/')
            depth = len(parts) - 1
            indent = '----' * depth
            file_name = parts[-1]
            
            checkbox = widgets.Checkbox(
                value=False,
                description=f'{indent}📄 {file}',
                style={'description_width': 'initial'}
            )
            checkbox_widgets.append((checkbox, file))
            
            html += f"<div>{indent}📄 {file_name}</div>"
    
    html += "</div>"
    return html, checkbox_widgets

# 全選択ボタン
def select_all(b):
    for checkbox, _ in file_checkboxes:
        checkbox.value = True

# 全解除ボタン
def deselect_all(b):
    for checkbox, _ in file_checkboxes:
        checkbox.value = False

# ダウンロードコマンド表示関数（チェック状態を維持）
def show_download_commands(b):
    selected_files = [file for checkbox, file in file_checkboxes if checkbox.value]
    
    if not selected_files:
        print("❌ Select files to download.")
        return
    
    repo_id = repo_input.value.strip()
    wget_cmds, curl_cmds = generate_download_commands(repo_id, selected_files)
    
    # ダウンロードコマンド表示エリア
    with download_output:
        download_output.clear_output()
        display(HTML("<h3>📥 Commands to Download</h3>"))
        
        # Wgetコマンド
        wget_section = widgets.Accordion([widgets.Textarea(value=wget_cmds, description='Wget:', layout={'width': '100%'})])
        wget_section.set_title(0, 'Wget')
        display(wget_section)
        
        # Curlコマンド
        curl_section = widgets.Accordion([widgets.Textarea(value=curl_cmds, description='Curl:', layout={'width': '100%'})])
        curl_section.set_title(0, 'Curl')
        display(curl_section)
    

# ボタンクリック時の処理
def on_button_click(b):
    with output:
        output.clear_output()
        repo_id = repo_input.value.strip()
        
        if not repo_id:
            print("❌ Input repository name.")
            return
        
        try:
            print(f"📂 Fetching the file list of {repo_id} ...")
            
            files = list_repo_files(repo_id=repo_id, revision='main')
            
            print(f"✅ Done! The Number of files: {len(files)}")
            
            # ツリーとチェックボックスを作成
            tree_html, checkboxes = create_tree_with_checkboxes(files)
            file_checkboxes.clear()
            file_checkboxes.extend(checkboxes)
            
            display(HTML(tree_html))
            for checkbox, _ in checkboxes:
                display(checkbox)
            
            # 全選択・全解除ボタン
            button_box = widgets.HBox([
                widgets.Button(description='🔘 Select All', tooltip='select all items'),
                widgets.Button(description='⚪️ Deselect All', tooltip='deselect all items')
            ])
            
            button_box.children[0].on_click(select_all)
            button_box.children[1].on_click(deselect_all)
            display(button_box)
            
            # ダウンロードコマンド生成ボタン
            download_btn = widgets.Button(description='📥 Generate Download Commands')
            download_btn.on_click(show_download_commands)
            display(download_btn)
            
        except Exception as e:
            print(f"❌ An erorr occured: {e}")

button.on_click(on_button_click)



In [4]:
display(widgets.VBox([repo_input, button]))

In [5]:
display(output)

Output()

In [6]:
display(download_output)

Output()